In [1]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import time
import re

# --- 設定 ---
DB_NAME = "google_repos.db"
TARGET_URL = "https://github.com/google?tab=repositories"

def main():
    # 1. データベースの初期化
    setup_database()
    
    # 2. スクレイピング実行
    data = scrape_github_repos()
    
    # 3. データの保存
    save_to_db(data)
    
    # 4. 保存結果の表示
    show_saved_data()

def setup_database():
    """DBとテーブルを作成し、既存データをクリアする"""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS repositories (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            language TEXT,
            stars TEXT
        )
    ''')
    cursor.execute('DELETE FROM repositories') # 毎回クリーンな状態で始める
    conn.commit()
    conn.close()
    print(f"[INFO] データベース {DB_NAME} を初期化しました。")

def scrape_github_repos():
    """Githubからリポジトリ情報を取得する"""
    print(f"[INFO] {TARGET_URL} にアクセス中...")
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
    }
    
    try:
        response = requests.get(TARGET_URL, headers=headers)
        response.raise_for_status()
    except Exception as e:
        print(f"[ERROR] アクセス失敗: {e}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    results = []
    
    # Githubの標準的なリストアイテム（liタグ）を取得
    # itemprop="owns" はGithubがリポジトリリストに使っている属性です
    repo_list = soup.find_all('li', itemprop='owns')

    # もし見つからなければ、h3タグベースの検索に切り替える（予備策）
    if not repo_list:
        print("[INFO] 標準リストが見つかりません。見出し検索モードに切り替えます。")
        repo_list = soup.find_all('li', class_='col-12') # 一般的なリストクラス

    print(f"[INFO] {len(repo_list)} 件の要素を解析します。")

    for item in repo_list:
        try:
            # --- 1. リポジトリ名 ---
            name_tag = item.find('a', itemprop='name codeRepository')
            if not name_tag:
                # 予備: h3の中のaタグを探す
                h3 = item.find('h3')
                name_tag = h3.find('a') if h3 else item.find('a')
            
            if not name_tag:
                continue

            name = name_tag.get_text(strip=True)
            
            # --- 2. 言語 ---
            lang_tag = item.find('span', itemprop='programmingLanguage')
            language = lang_tag.get_text(strip=True) if lang_tag else "No Language"
            
            # --- 3. スター数 ---
            # リンクが /stargazers で終わるものを探す
            star_tag = item.find('a', href=re.compile(r'/stargazers$'))
            if star_tag:
                # "3,400" -> "3400" に整形
                stars = star_tag.get_text(strip=True).replace(',', '')
            else:
                stars = "0"

            # 結果をリストに追加
            results.append((name, language, stars))
            print(f"[SCRAPED] {name} | {language} | ⭐ {stars}")
            
            # 【重要】要件：time.sleep(1)を入れる
            time.sleep(1)

        except Exception as e:
            print(f"[WARN] 解析エラー: {e}")
            continue

    return results

def save_to_db(data):
    """取得したデータをDBに保存"""
    if not data:
        print("[WARN] 保存するデータがありません。")
        return

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.executemany('INSERT INTO repositories (name, language, stars) VALUES (?, ?, ?)', data)
    conn.commit()
    conn.close()
    print(f"[INFO] {len(data)} 件のデータを保存しました。")

def show_saved_data():
    """保存データをSELECT文で表示"""
    print("\n--- DB保存結果 (SELECT * FROM repositories) ---")
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    cursor.execute('SELECT * FROM repositories')
    rows = cursor.fetchall()
    
    if not rows:
        print("データなし")
    else:
        # ヘッダー表示
        print(f"{'ID':<4} | {'Name':<30} | {'Language':<15} | {'Stars':<8}")
        print("-" * 65)
        for row in rows:
            # ID, Name, Language, Stars
            print(f"{row[0]:<4} | {row[1]:<30} | {row[2]:<15} | {row[3]:<8}")

    conn.close()

if __name__ == "__main__":
    main()

[INFO] データベース google_repos.db を初期化しました。
[INFO] https://github.com/google?tab=repositories にアクセス中...
[INFO] 標準リストが見つかりません。見出し検索モードに切り替えます。
[INFO] 6 件の要素を解析します。
[SCRAPED] material-design-icons | No Language | ⭐ 52.6k
[SCRAPED] guava | Java | ⭐ 51.3k
[SCRAPED] zx | JavaScript | ⭐ 44.9k
[SCRAPED] styleguide | HTML | ⭐ 38.7k
[SCRAPED] leveldb | C++ | ⭐ 38.4k
[SCRAPED] googletest | C++ | ⭐ 37.5k
[INFO] 6 件のデータを保存しました。

--- DB保存結果 (SELECT * FROM repositories) ---
ID   | Name                           | Language        | Stars   
-----------------------------------------------------------------
1    | material-design-icons          | No Language     | 52.6k   
2    | guava                          | Java            | 51.3k   
3    | zx                             | JavaScript      | 44.9k   
4    | styleguide                     | HTML            | 38.7k   
5    | leveldb                        | C++             | 38.4k   
6    | googletest                     | C++             | 37.5k   
